# Building chroma_v3

Embeds the split corpus into the vector store the app ships with. It runs on Colab because
embedding this many chunks needs a GPU.

**It survives a disconnect.** The store is written straight to Google Drive and every chunk
has a fixed ID, so re-running the embedding cell picks up where it stopped. Nothing is
embedded twice, and nothing already on disk is overwritten.

**Before you start**

1. Upload `document_splits_v3.json.gz` to the root of your Drive.
2. Runtime → Change runtime type → T4 GPU.

**What v3 added over v2**

| | v2 | v3 |
|---|---|---|
| Laws | 588 | 3,075 |
| Cassation cases | 9,001 | 9,009 |
| Chunks in the splits file | 25,738 | 47,383 |

The Civil Code (1,057 articles) and the Evidence Law (158) were missing from v2 altogether,
along with 709 amendments. Every legislation chunk now carries its law name and chapter
heading as a prefix.

**This notebook does not reproduce the released store.** It builds 47,383 chunks, which is
what `document_splits_v3.json.gz` contains. The `chroma_v3` the app ships with holds 49,782.
The documents are the same in both — 3,075 laws, 9,009 cases, 93 rulings — but in the
released store long judgments are split into several chunks each, where the splits file
keeps one chunk per case. That extra pass was run once and its parameters were not kept, so
it cannot be repeated from anything in this repository.

## 1. Check the GPU

This has to list a GPU. If it errors: Runtime → Change runtime type → T4 GPU.

In [ ]:
!nvidia-smi

## 2. Install dependencies

In [ ]:
!pip install -q langchain-chroma langchain-huggingface sentence-transformers

## 3. Mount Drive and load the corpus

The assert stops the notebook if the file is not the one expected.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import gzip, json, collections
from langchain_core.documents import Document

SRC = '/content/drive/MyDrive/document_splits_v3.json.gz'

# What document_splits_v3.json.gz actually contains. The store the app ships with
# holds 49,782, because long judgments were split further in a later pass that is
# not in this notebook -- see the note under 'What v3 added over v2' above.
EXPECTED_CHUNKS = 47383
with gzip.open(SRC, 'rt', encoding='utf-8') as f:
    recs = json.load(f)

splits = [Document(page_content=r['page_content'], metadata=r['metadata']) for r in recs]
# Fixed IDs make the run idempotent: adding the same chunk twice cannot duplicate it.
ids = [f'c{i:06d}' for i in range(len(splits))]

print('chunks loaded:', len(splits))
print('by source    :', dict(collections.Counter(d.metadata['source'] for d in splits)))
assert len(splits) == EXPECTED_CHUNKS, f'expected {EXPECTED_CHUNKS}, got {len(splits)}'
print('OK')

## 4. Load the embedding model

The same model as v2, `BAAI/bge-m3`. It has to be the same one, or the new vectors would not
be comparable with the old.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"batch_size": 64},
)
print('embedding model ready')

## 5. Embed into Drive

Re-run this cell if the session drops.

It asks the store how many chunks it already holds and starts from there, so re-running after
a disconnect resumes and re-running after a completed pass does nothing.

Expect 45 to 90 minutes on a T4. It is slower than writing to local disk because Drive is
network-mounted, which is the price of being resumable.

The red `Token indices sequence length is longer than...` warnings are the tokenizer noting
long chunks, and are expected.

In [ ]:
import time
from langchain_chroma import Chroma

PERSIST = '/content/drive/MyDrive/chroma_v3'   # on Drive -> survives disconnects
BATCH   = 2000                                 # checkpoint every 2,000 chunks

vectordb = Chroma(persist_directory=PERSIST, embedding_function=embedding)
already = vectordb._collection.count()
print(f'already stored: {already:,} / {len(splits):,}')

if already >= len(splits):
    print('nothing to do - already complete')
else:
    t0 = time.time()
    for i in range(already, len(splits), BATCH):
        b_docs = splits[i:i + BATCH]
        b_ids  = ids[i:i + BATCH]
        vectordb.add_documents(documents=b_docs, ids=b_ids)
        done = i + len(b_docs)
        el   = time.time() - t0
        rate = (done - already) / el if el else 0
        eta  = (len(splits) - done) / rate / 60 if rate else 0
        print(f'{done:>6,}/{len(splits):,}   {el/60:5.1f} min elapsed   ~{eta:4.1f} min left',
              flush=True)
    print('EMBEDDING DONE in', round((time.time()-t0)/60, 1), 'min')

## 6. Verify the count

This is the check that matters. It should equal the count asserted in step 3 — 47,383 for
`document_splits_v3.json.gz`. If it is short, re-run the embedding cell and it will resume.
If it stays short after a complete pass, the corpus file itself is wrong.

In [ ]:
count = vectordb._collection.count()
print('FINAL COUNT:', count, 'of', len(splits))
assert count == len(splits), f'MISMATCH: stored {count}, expected {len(splits)} - re-run cell 5'
print('OK - all chunks stored')

## 7. Smoke test

These are questions the earlier store failed on. What we want to see is `lloc` rows, and
`L1901` — the Civil Code — for عيوب الرضاء in particular. Against v2 that question returned no
legislation at all, only court cases.

In [ ]:
tests = [
    'ما هي عيوب الرضاء في القانون المدني؟',
    'ما هو السبب في العقد؟',
    'ما هي عقوبة الرشوة؟',
    'ما هي قواعد الاثبات في المواد المدنية والتجارية؟',
]
for q in tests:
    print('=' * 80)
    print('Q:', q)
    for doc, score in vectordb.similarity_search_with_score(q, k=4):
        m = doc.metadata
        print(f"   {score:.3f}  {m['source']:5} {str(m.get('doc_id'))[:14]:15} "
              f"art={str(m.get('article_no'))[:5]:5} {doc.page_content[:65].strip()}")

## 8. Zip it for download

The store is already safe on Drive. This just packs it into a single file that is easier to
pull down and drop into `data/chroma_v3`.

In [ ]:
!zip -r -q /content/chroma_v3.zip /content/drive/MyDrive/chroma_v3
!cp /content/chroma_v3.zip /content/drive/MyDrive/
!ls -lh /content/drive/MyDrive/chroma_v3.zip
print('saved - download chroma_v3.zip from your Drive')